# SmartTraffic AI — Google Colab End-to-End
**Custom YOLO11 helmet/no-helmet training + motorcycle ByteTrack + video evidence**

**Before running:** Colab → Runtime → Change runtime type → **T4 GPU**. Upload a CCTV MP4 to Colab when prompted. The notebook downloads a **motorcycle-rider helmet dataset** using your Roboflow API key; check class labels before training. **No pretrained helmet weights are required.**

This notebook is a reproducible training pipeline, not a claim that a model is already trained. Detection quality depends on the dataset, labels, and camera angle. YOLO11n is initialized from general pretrained weights, then **fine-tuned on your custom helmet dataset**. For random initialization, set `BASE_MODEL='yolo11n.yaml'` (slower and typically less accurate on small datasets).

## 1 — Install dependencies and verify GPU

In [ ]:
!pip -q install "ultralytics>=8.3,<9" "roboflow>=1.1" "opencv-python-headless>=4.8" pyyaml pandas
import torch, os
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE — enable T4 GPU')
assert torch.cuda.is_available(), 'Enable GPU in Runtime settings before training'
from pathlib import Path
ROOT=Path('/content/SmartTrafficAI'); ROOT.mkdir(exist_ok=True)


## 2 — Download the dataset from Roboflow
Open [Motorcycle Helmet Detection](https://universe.roboflow.com/innovatech/motorcycle-helmet-q0wmd) and verify the dataset contains **motorcycle helmets and unhelmeted riders**, not hardhats. Find its exact **workspace / project / version** in Roboflow’s *Download Dataset → Show download code*; paste those values below. Your API key is requested privately and not saved in this notebook.

In [ ]:
from getpass import getpass
from roboflow import Roboflow
RF_WORKSPACE = 'innovatech'  # change if dataset page shows another workspace
RF_PROJECT = 'motorcycle-helmet-q0wmd'  # check dataset download code
RF_VERSION = 1  # IMPORTANT: replace with an available dataset version
RF_KEY = getpass('Roboflow API key (hidden): ')
rf=Roboflow(api_key=RF_KEY)
project=rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset=project.version(RF_VERSION).download('yolov8', location=str(ROOT/'dataset'), overwrite=True)
print('Dataset:',dataset.location)


## 3 — Validate dataset and classes
**Stop here** if labels do not describe helmet vs no-helmet on motorcycle riders. Different Roboflow projects use different class names; do not assume class IDs.

In [ ]:
import yaml
from pathlib import Path
DATA_YAML=Path(dataset.location)/'data.yaml'
assert DATA_YAML.exists(), f'Missing {DATA_YAML}'
data=yaml.safe_load(DATA_YAML.read_text())
names=data['names']
if isinstance(names,dict): names=[names[k] for k in sorted(names,key=lambda x:int(x))]
print('Dataset class names:',names)
print('Train:',data.get('train'),'Valid:',data.get('val'))
assert len(names)>=2, 'Need helmet and no-helmet classes. Choose a different dataset.'
# Override these only after inspecting names above
HELMET_CLASS = next((i for i,x in enumerate(names) if 'helmet' in x.lower() and not any(t in x.lower().replace(' ','_') for t in ('no_','without','non_','not_'))),None)
NO_HELMET_CLASS = next((i for i,x in enumerate(names) if any(t in x.lower().replace(' ','_') for t in ('no_helmet','without_helmet','non_helmet','not_helmet'))),None)
print('Guessed helmet class:',HELMET_CLASS,'no-helmet class:',NO_HELMET_CLASS)
assert HELMET_CLASS is not None and NO_HELMET_CLASS is not None and HELMET_CLASS!=NO_HELMET_CLASS, 'Manually set HELMET_CLASS and NO_HELMET_CLASS after inspecting labels; do not train with incorrect class IDs.'


## 4 — Custom training (YOLO11n)

In [ ]:
from ultralytics import YOLO
BASE_MODEL='yolo11n.pt'  # fine-tune; for from-random-weights use yolo11n.yaml
model=YOLO(BASE_MODEL)
results=model.train(data=str(DATA_YAML),epochs=60,imgsz=640,batch=8,device=0,workers=2,patience=15,project=str(ROOT/'runs'),name='helmet_custom',exist_ok=True,seed=42,plots=True)
BEST=ROOT/'runs'/'helmet_custom'/'weights'/'best.pt'
assert BEST.exists(), 'Training did not create best.pt'
print('Custom model saved:',BEST)


## 5 — Evaluate on validation set

In [ ]:
helmet_model=YOLO(str(BEST))
metrics=helmet_model.val(data=str(DATA_YAML),split='val',device=0,plots=True)
print('mAP50:',metrics.box.map50,'mAP50-95:',metrics.box.map)


## 6 — Upload CCTV video

In [ ]:
from google.colab import files
uploaded=files.upload()
assert uploaded,'Upload a CCTV MP4'
VIDEO=Path('/content')/next(iter(uploaded))
assert VIDEO.exists()
print(VIDEO)


## 7 — Run V2 tracking and helmet violation detection

Upload the **raw, unannotated** MP4. The tracker assigns motorcycle IDs; helmet-to-bike association is heuristic, so review evidence manually. The Python script is bundled under `src/` in this repository.


In [ ]:
# In Colab, upload src/smarttraffic_v2.py once, or clone this repository.
from pathlib import Path
import subprocess, sys
SCRIPT = Path('/content/smarttraffic_v2.py')
if not SCRIPT.exists():
    repo_script = Path('/content/SmartTrafficAI_GitHub/src/smarttraffic_v2.py')
    if repo_script.exists(): SCRIPT = repo_script
assert SCRIPT.exists(), 'Upload src/smarttraffic_v2.py or clone the repository into /content/SmartTrafficAI_GitHub'
OUTPUT_V2 = Path('/content/SmartTrafficAI_V2')
cmd = [sys.executable, str(SCRIPT), '--video', str(VIDEO), '--helmet-model', str(BEST),
       '--output', str(OUTPUT_V2), '--bike-conf', '0.22', '--helmet-conf', '0.30',
       '--confirm-frames', '4', '--imgsz', '960']
subprocess.run(cmd, check=True)
print('Generated files:', [str(p) for p in OUTPUT_V2.rglob('*') if p.is_file()])


## 8 — Preview results and inspect violations


In [ ]:
from IPython.display import Video, display
import pandas as pd
videos = sorted(OUTPUT_V2.rglob('*.mp4'))
assert videos, f'No output MP4 found under {OUTPUT_V2}'
result_video = max(videos, key=lambda p:p.stat().st_size)
print('Preview:', result_video)
display(Video(str(result_video), embed=True, width=900))
for report in OUTPUT_V2.rglob('*.csv'):
    print('Report:', report)
    display(pd.read_csv(report))


## 9 — Export the results


In [ ]:
from google.colab import files
import shutil
archive = shutil.make_archive('/content/SmartTrafficAI_V2_Results','zip',str(OUTPUT_V2))
files.download(archive)


## Notes and limitations

- The helmet detector is fine-tuned from pretrained YOLO11n weights, **not trained from random initialization**.
- The dataset is retrieved from Roboflow; confirm dataset version, license and class definitions before training or redistribution.
- Validation metrics reflect the chosen validation split, not real-world CCTV accuracy.
- Track IDs can fragment and head-to-motorcycle association can be wrong in crowded scenes.
- Synthetic traffic footage is for demonstration, not a real-world benchmark.
- Review evidence and CSV before claiming a violation.
